In [ ]:
import os, sys, shutil, subprocess, json, platform
from pathlib import Path
import psutil, numpy, scipy, sklearn, pandas, pyarrow

print("virtual_memory:", psutil.virtual_memory())
print("cpu_count:", psutil.cpu_count())
print("versions:", {"python": platform.python_version(), "numpy": numpy.__version__, "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "pandas": pandas.__version__, "pyarrow": pyarrow.__version__, "psutil": psutil.__version__})
def find_one(name):
    hits = sorted(Path("/kaggle/input").rglob(name))
    print("found", name, [str(h) for h in hits])
    if len(hits) != 1:
        raise FileNotFoundError(f"expected exactly one {name}, found {len(hits)}")
    return hits[0]
package = find_one("r20_cpu_cv.py").parent
base_dir = find_one("nlm_pool_v5.parquet").parent
open_dir = find_one("ntl_open_pool.parquet").parent
working = Path("/kaggle/working")
for name in ("r19_svc_component.py", "r19_overlap_audit.py", "r19_cpu_cv.py", "r20_cpu_cv.py"):
    shutil.copy2(package / name, working / name)
available_gib = psutil.virtual_memory().available / 1024**3
minimum_gib = max(22.0, min(24.0, available_gib - 1.0))
os.environ.update({
    "NTL_DEV": str(package / "development.json"),
    "NTL_BASE_DIR": str(base_dir),
    "NTL_OPEN_DIR": str(open_dir),
    "NTL_R20_OUT": str(working / "r20_run"),
    "NTL_ALLOW_ENV_VERSIONS": "1",
    "NTL_R20_LIMIT_SECONDS": "32400",
    "NTL_R20_MIN_AVAILABLE_GIB": str(minimum_gib),
    "NTL_R20_MAX_RSS_GIB": "28",
})
print("minimum available GiB:", minimum_gib)
output = working / "r20_out"
output.mkdir(exist_ok=True)
try:
    subprocess.run([sys.executable, str(working / "r20_cpu_cv.py"), "--run"], check=True, cwd=working)
finally:
    run_dir = working / "r20_run"
    for name in ("result.json", "receipt.json", "claim.json", "worker.log"):
        if (run_dir / name).is_file():
            shutil.copy2(run_dir / name, output / name)
    if (output / "receipt.json").is_file():
        print("receipt.json:", (output / "receipt.json").read_text(encoding="utf-8"))
